# 04 分区建模 + 空间交叉验证 + 模型对比（任务书核心）

In [ ]:
import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
FIG = ensure_dir(OUT/'figures')
df = pd.read_parquet(OUT/'bcf_final.parquet')


In [ ]:
from src.spatial_cv import make_spatial_groups, get_group_kfold
from src.models import fit_oof_with_spatialcv
from src.utils import safe_col
from src.plots import obs_pred_scatter, residual_plot

bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
max_rows = int(cfg['modeling']['max_train_rows'])
df_run = df.sample(max_rows, random_state=cfg['project']['seed']).reset_index(drop=True) if len(df)>max_rows else df.reset_index(drop=True)

cv = get_group_kfold(cfg)

group_keys = ['crop','ph_bin'] if 'crop' in df_run.columns else ['ph_bin']
models = cfg['modeling']['models']
rows = []

for key, sub in df_run.groupby(group_keys, dropna=False):
    sub = sub.reset_index(drop=True)
    gsub = make_spatial_groups(sub, cfg)

    for m in models:
        res = fit_oof_with_spatialcv(sub, cfg, gsub, m, cv)
        title = f"{key} | {m} | r2={res.metrics['r2']:.3f}, rmse={res.metrics['rmse']:.3f}"
        key_str = "_".join(map(str, key if isinstance(key, tuple) else (key,)))
        obs_pred_scatter(res.y_true, res.oof_pred, title, str(FIG/f"obs_pred_{key_str}_{m}.png"))
        rows.append({"group": str(key), "model": m, **res.metrics})

metrics = pd.DataFrame(rows).sort_values(['group','model'])
metrics.to_excel(OUT/'metrics_spatialcv.xlsx', index=False)
metrics.head(20)


# 额外输出残差图（更直观诊断系统性偏差）
resid_path = fig_dir / f"resid_{crop}_{ph}_{model}.png"
residual_plot(y_true, y_pred, title=title.replace('obs-pred','residual'), out_png=resid_path)
